# UAE Mobile Intelligence - H3 Resolution Choice

The brief requires the geographic unit for zone aggregation to be chosen empirically (measurement
density + score stability), not just taken on faith from its own resolution-7 suggestion, and to
reserve resolution 8 for display only. This notebook compares resolutions 6, 7 and 8 on the
UAE-clipped, 8-quarter Ookla tile dataset built by
[`01_ookla_collection.ipynb`](01_ookla_collection.ipynb), and records the chosen resolution for
downstream notebooks (`05_zone_aggregation.ipynb`, `06_first_uae_map.ipynb`) to reuse.

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import h3
import numpy as np
import pandas as pd

## Load the processed tiles

Drop the geometry columns here -- H3 cell assignment works directly off `tile_y`/`tile_x`
(latitude/longitude), and we don't need the polygon geometries for this comparison.

In [2]:
tiles = gpd.read_parquet("../data/processed/ookla_tiles_uae.parquet")
tiles = pd.DataFrame(tiles.drop(columns=["geometry", "tile_geometry"]))

print("Tile-quarter rows:", len(tiles))
print("Quarters:", sorted(tiles["quarter"].unique()))

Tile-quarter rows: 50860
Quarters: ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']


## Assign every tile to an H3 cell at each candidate resolution

`h3.latlng_to_cell` takes `(lat, lon)` -- that's `(tile_y, tile_x)` for this dataset, the same
swap gotcha documented in [`00_ookla_dataset_overview.ipynb`](00_ookla_dataset_overview.ipynb).
Get this backwards and every zone lands silently in the sea.

In [3]:
CANDIDATE_RESOLUTIONS = [6, 7, 8]

for res in CANDIDATE_RESOLUTIONS:
    tiles[f"h3_{res}"] = [
        h3.latlng_to_cell(lat, lon, res)
        for lat, lon in zip(tiles["tile_y"], tiles["tile_x"])
    ]

tiles[["quadkey", "tile_x", "tile_y", "h3_6", "h3_7", "h3_8"]].head()

,quadkey,tile_x,tile_y,h3_6,h3_7,h3_8
0,1230231133033122,56.055,25.9803,8643add37ffffff,8743add36ffffff,8843add363fffff
1,1230231133033130,56.066,25.9852,8643add37ffffff,8743add30ffffff,8843add305fffff
2,1230231133033132,56.066,25.9803,8643add37ffffff,8743add32ffffff,8843add32bfffff
3,1230231133033310,56.066,25.9753,8643add37ffffff,8743add32ffffff,8843add321fffff
4,1230231133033322,56.055,25.9605,8643aca4fffffff,8743aca4dffffff,8843aca4d3fffff


## Measurement density, per resolution

Snapshot on the latest quarter (2026Q2), since that's what a live map would show today. For each
resolution: how many zones does the UAE resolve into, how many tiles/tests pool into each zone, and
what share of zones clear a minimum evidence bar (>=30 tests, the threshold used later for the
Confidence Score work)?

In [4]:
LATEST_QUARTER = "2026Q2"
latest = tiles[tiles["quarter"] == LATEST_QUARTER]

density_rows = []
for res in CANDIDATE_RESOLUTIONS:
    col = f"h3_{res}"
    g = latest.groupby(col).agg(
        n_tiles=("quadkey", "count"),
        tests=("tests", "sum"),
        devices=("devices", "sum"),
    )
    density_rows.append({
        "resolution": res,
        "zones (latest qtr)": len(g),
        "zones (all 8 qtrs)": tiles[col].nunique(),
        "tiles/zone (median)": g["n_tiles"].median(),
        "tiles/zone (mean)": round(g["n_tiles"].mean(), 2),
        "tests/zone (median)": g["tests"].median(),
        "tests/zone (mean)": round(g["tests"].mean(), 2),
        "share tests>=30": round((g["tests"] >= 30).mean(), 3),
        "share single-tile zones": round((g["n_tiles"] == 1).mean(), 3),
    })

density = pd.DataFrame(density_rows).set_index("resolution")
density

,zones (latest qtr),zones (all 8 qtrs),tiles/zone (median),tiles/zone (mean),tests/zone (median),tests/zone (mean),share tests>=30,share single-tile zones
resolution,,,,,,,,
6,671,1074,3.0,10.25,5.0,61.44,0.231,0.311
7,1815,3400,2.0,3.79,4.0,22.71,0.173,0.407
8,4821,9859,1.0,1.43,3.0,8.55,0.060,0.651


**Sanity check against the brief:** at resolution 7 the median zone has 4 tests and ~83% of
zones fall below 30 tests -- the brief cites these exact figures ("the median zone has only 4 tests
per quarter... about 83% of zones fall below 30 tests") as the numbers its own figures are based on.
Matching them here on independently-rebuilt UAE-clipped data is a strong empirical confirmation that
resolution 7 is the unit the brief means, not just an assumption to take on faith.

## Score stability, per resolution

A zone-level average is only useful if it doesn't swing wildly quarter to quarter purely from
sampling noise. For every resolution, compute the test-weighted average download per zone per
quarter, then the quarter-over-quarter relative change for zones present in both quarters of each
consecutive pair. Finer resolutions pool fewer tiles per zone, so their zone averages should be
visibly noisier (wider spread of relative change) than coarser ones.

In [5]:
QUARTERS_ORDER = ["2024Q3", "2024Q4", "2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2"]

tiles["w_download"] = tiles["avg_d_kbps"] * tiles["tests"]

stability_rows = []
for res in CANDIDATE_RESOLUTIONS:
    col = f"h3_{res}"

    zq = tiles.groupby([col, "quarter"]).agg(
        tests=("tests", "sum"),
        w_download=("w_download", "sum"),
    ).reset_index()
    zq["avg_download_kbps"] = zq["w_download"] / zq["tests"]

    pivot = zq.pivot(index=col, columns="quarter", values="avg_download_kbps")
    pivot = pivot[QUARTERS_ORDER]

    rel_changes = []
    for q0, q1 in zip(QUARTERS_ORDER[:-1], QUARTERS_ORDER[1:]):
        both = pivot[[q0, q1]].dropna()
        rel_changes.append((both[q1] - both[q0]) / both[q0])

    rc = pd.concat(rel_changes).replace([np.inf, -np.inf], np.nan).dropna()

    stability_rows.append({
        "resolution": res,
        "zone-quarter pairs": len(rc),
        "QoQ change (median)": round(rc.median(), 3),
        "QoQ change (IQR)": round(rc.quantile(0.75) - rc.quantile(0.25), 3),
    })

stability = pd.DataFrame(stability_rows).set_index("resolution")
stability

,zone-quarter pairs,QoQ change (median),QoQ change (IQR)
resolution,,,
6,3412,0.013,0.982
7,8570,0.031,1.036
8,20854,0.029,1.348


The interquartile range of quarter-over-quarter relative change widens monotonically from
resolution 6 to 8 -- exactly the direction the brief predicts: coarser zones pool more tiles, so
their averages are less exposed to single-test sampling noise. (Raw standard deviation is dominated
by a handful of zones with tiny denominators exploding the relative change, which is why IQR --
robust to those outliers -- is the number to read here.)

## Decision

| Criterion | Res 6 | Res 7 | Res 8 |
|---|---|---|---|
| Zones nationally (latest qtr) | too few for zone-level drill-down granularity | matches the brief's own scale | far too sparse -- 65% single-tile zones |
| Density (tests/zone, share >=30) | best-supported, but coarse | matches the brief's own cited figures almost exactly | "almost no aggregation," as the brief predicts |
| Stability (QoQ IQR) | most stable | close behind res 6, clearly better than res 8 | worst -- least trustworthy zone averages |
| Zones clearing evidence bar | ~155 zones | ~314 zones -- matches "a few hundred well-measured zones nationally" | ~289 zones, but out of ~4,800 total (mostly noise) |

Resolution 7 is the chosen unit: it reproduces the brief's own density figures on independently
rebuilt data, gives a "few hundred well-measured zones nationally" evidence pool as promised, and
its stability is close to resolution 6 while supporting far more granular zone drill-down (1,815
vs. 671 zones nationally). Resolution 6 is not wrong, just coarser than needed; resolution 8 is
confirmed too sparse to aggregate meaningfully and is reserved for display only, exactly as the
brief suggests.

In [6]:
CHOSEN_RESOLUTION = 7

decision = {
    "chosen_resolution": CHOSEN_RESOLUTION,
    "latest_quarter_used": LATEST_QUARTER,
    "zones_latest_quarter": int(density.loc[CHOSEN_RESOLUTION, "zones (latest qtr)"]),
    "zones_all_quarters": int(density.loc[CHOSEN_RESOLUTION, "zones (all 8 qtrs)"]),
    "median_tests_per_zone": float(density.loc[CHOSEN_RESOLUTION, "tests/zone (median)"]),
    "share_zones_tests_ge_30": float(density.loc[CHOSEN_RESOLUTION, "share tests>=30"]),
    "qoq_download_change_iqr": float(stability.loc[CHOSEN_RESOLUTION, "QoQ change (IQR)"]),
    "justification": (
        "Resolution 7 reproduces the brief's own cited density figures (median 4 tests/zone, "
        "~83% of zones below 30 tests) on independently UAE-clipped data, yields a 'few hundred "
        "well-measured zones nationally' evidence pool consistent with the brief's expectation, "
        "and has quarter-over-quarter score stability close to resolution 6 while resolving far "
        "more granular zones for drill-down. Resolution 8 is confirmed too sparse to aggregate "
        "(mean 1.4 tiles/zone) and is reserved for display only."
    ),
}

out_path = Path("../data/processed/h3_resolution.json")
out_path.write_text(json.dumps(decision, indent=2))

print("Saved decision to:", out_path)
print(json.dumps(decision, indent=2))

Saved decision to: ..\data\processed\h3_resolution.json
{
  "chosen_resolution": 7,
  "latest_quarter_used": "2026Q2",
  "zones_latest_quarter": 1815,
  "zones_all_quarters": 3400,
  "median_tests_per_zone": 4.0,
  "share_zones_tests_ge_30": 0.173,
  "qoq_download_change_iqr": 1.036,
  "justification": "Resolution 7 reproduces the brief's own cited density figures (median 4 tests/zone, ~83% of zones below 30 tests) on independently UAE-clipped data, yields a 'few hundred well-measured zones nationally' evidence pool consistent with the brief's expectation, and has quarter-over-quarter score stability close to resolution 6 while resolving far more granular zones for drill-down. Resolution 8 is confirmed too sparse to aggregate (mean 1.4 tiles/zone) and is reserved for display only."
}
